# **JOB MATCHING - MACHINE LEARNING MODEL**



import

In [33]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

import pandas as pd
import numpy as np
import re
import warnings
import textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import os
os.makedirs("saved_model", exist_ok=True)
os.makedirs("plots", exist_ok=True)

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score,
    classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
plt.rcParams.update({
    "figure.facecolor": "#0F1117",
    "axes.facecolor": "#161B22",
    "axes.edgecolor": "#30363D",
    "axes.labelcolor": "#C9D1D9",
    "xtick.color": "#8B949E",
    "ytick.color": "#8B949E",
    "text.color": "#C9D1D9",
    "grid.color": "#21262D",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
    "font.family": "DejaVu Sans",
    "font.size": 11,
})

PALETTE = ["#58A6FF", "#3FB950", "#F78166", "#D2A8FF", "#FFA657",
           "#79C0FF", "#56D364", "#FF7B72", "#BC8CFF", "#FFD580"]
ACCENT = "#58A6FF"
ACCENT2 = "#3FB950"
BG_DARK = "#0F1117"
BG_MID = "#161B22"
TEXT_MAIN = "#C9D1D9"
DIV = "=" * 70


def section(t):
    print(f"\n{DIV}\n  {t}\n{DIV}")


def sub(t):
    print(f"\n{'-' * 60}\n  {t}\n{'-' * 60}")



# LOAD & PREPARE DATA

In [36]:
section("LOAD & PREPARE DATA")

print("[INFO] Membaca dataset ...")
df_jobs = pd.read_csv("job_postings.csv", engine='python', on_bad_lines='skip')
df_js = pd.read_csv("jobstreet_all_job_dataset.csv", engine='python', on_bad_lines='skip')
print(f"  job_postings : {df_jobs.shape[0]:,} baris | jobstreet : {df_js.shape[0]:,} baris")


  LOAD & PREPARE DATA
[INFO] Membaca dataset ...
  job_postings : 11,696 baris | jobstreet : 21,796 baris


Stop words

In [37]:
BAHASA_SW = {
    "saya","anda","yang","dan","di","ini","itu","dengan","untuk","dari","dalam",
    "pada","ke","adalah","tidak","akan","juga","sudah","belum","bisa","ada",
    "kami","kita","mereka","oleh","sebagai","atau","jika","maka","dapat","telah",
    "lebih","sangat","serta","setelah","antara","secara","sesuai","sehingga",
    "harus","perlu","mampu","baik","setiap","berbagai","terhadap","bidang",
    "tahun","pengalaman","lulusan","kemampuan","kerja","perusahaan","tim",
    "bekerja","menggunakan","memiliki","terampil","berpengalaman","terbiasa",
    "familiar","mahir","pernah","siap","kepada","iaitu","bagi","semua","boleh",
    "sahaja","hanya","beliau","lain","lagi","pula","tetapi","namun","kerana",
    "apabila","mana","selepas","sebelum","melalui","seperti","supaya",
}
ALL_SW = list(ENGLISH_STOP_WORDS | BAHASA_SW)

Master skill list

In [38]:
SKILLS = [
    "python","r","sql","java","javascript","typescript","scala","html","css",
    "react","node.js","php","c++","go","swift","machine learning","deep learning",
    "nlp","computer vision","tensorflow","keras","pytorch","scikit-learn","xgboost",
    "pandas","numpy","statistics","data analysis","data visualization",
    "data engineering","etl","feature engineering","tableau","power bi","looker",
    "excel","sap","spss","aws","gcp","azure","docker","kubernetes","git","linux",
    "spark","hadoop","airflow","kafka","project management","business analysis",
    "communication","leadership","teamwork","budgeting","reporting","agile",
    "microsoft office","accounting","audit","taxation","autocad",
    "electrical design","troubleshooting","maintenance","plc","arduino",
    "mechanical design","quality control","seo","social media","content writing",
    "google analytics","customer service","sales","negotiation",
]


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_skills(text: str) -> list:
    return [sk for sk in SKILLS if re.search(r"\b" + re.escape(sk) + r"\b", text)]

def compute_final_score(sim, skill_ratio, category_match):
    return (
        0.6 * sim +
        0.25 * skill_ratio +
        0.15 * category_match
    )

def infer_resume_category(skills: list) -> str:
    skills_set = set(skills)

    if skills_set & {"javascript", "react", "node.js", "java", "php", "html", "css"}:
        return "software"
    elif skills_set & {"python", "sql", "machine learning", "tableau", "power bi", "statistics", "data analysis"}:
        return "data"
    elif skills_set & {"accounting", "audit", "taxation", "sap", "budgeting"}:
        return "finance"
    elif skills_set & {"autocad", "plc", "electrical design", "maintenance", "arduino"}:
        return "engineering"
    elif skills_set & {"project management", "leadership", "business analysis", "reporting"}:
        return "management"
    else:
        return "other"


def infer_job_category(job_title: str) -> str:
    jt = str(job_title).lower()

    if any(k in jt for k in ["software", "developer", "engineer", "frontend", "backend", "fullstack", "programmer"]):
        return "software"
    elif any(k in jt for k in ["data", "analyst", "scientist", "business intelligence", "bi"]):
        return "data"
    elif any(k in jt for k in ["account", "finance", "tax", "auditor"]):
        return "finance"
    elif any(k in jt for k in ["electrical", "mechanical", "technician", "maintenance"]):
        return "engineering"
    elif any(k in jt for k in ["manager", "project manager", "supervisor", "product manager"]):
        return "management"
    else:
        return "other"


def get_category_match(resume_category: str, job_category: str) -> float:
    return 1.0 if resume_category == job_category else 0.0

job_postings

In [39]:
df_jp = df_jobs[[
    "job_id", "title", "description", "skills_desc",
    "formatted_experience_level", "formatted_work_type", "location"
]].copy()

df_jp.drop_duplicates("job_id", inplace=True)
df_jp["clean_text"] = (
    df_jp["title"].fillna("") + " " +
    df_jp["skills_desc"].fillna("") + " " +
    df_jp["description"].fillna("")
).apply(clean_text)

df_jp.rename(columns={
    "title": "job_title",
    "formatted_experience_level": "experience_level",
    "formatted_work_type": "work_type"
}, inplace=True)
df_jp["source"] = "job_postings"
df_jp["company"] = np.nan

jobstreet

In [40]:
df_js2 = df_js[[
    "job_id", "job_title", "company", "descriptions",
    "location", "category", "subcategory", "role", "type"
]].copy()

df_js2.drop_duplicates("job_id", inplace=True)
df_js2["clean_text"] = (
    df_js2["job_title"].fillna("") + " " +
    df_js2["category"].fillna("") + " " +
    df_js2["role"].fillna("") + " " +
    df_js2["descriptions"].fillna("")
).apply(clean_text)

df_js2.rename(columns={
    "descriptions": "description",
    "type": "work_type"
}, inplace=True)
df_js2["experience_level"] = np.nan
df_js2["source"] = "jobstreet"

COMMON = [
    "job_id", "job_title", "company", "description", "experience_level",
    "work_type", "location", "clean_text", "source"
]

df_combined = pd.concat([
    df_jp[[c for c in COMMON if c in df_jp.columns]],
    df_js2[[c for c in COMMON if c in df_js2.columns]],
], ignore_index=True)

SAMPLE_N = 20_000
if len(df_combined) > SAMPLE_N:
    df_combined = df_combined.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
print(f"[INFO] Dataset gabungan di-sample: {len(df_combined):,} lowongan")

print("[INFO] Mengekstrak skill dari job descriptions ...")
df_combined["job_skills"] = df_combined["clean_text"].apply(extract_skills)
df_combined["job_skill_count"] = df_combined["job_skills"].apply(len)


[INFO] Dataset gabungan di-sample: 20,000 lowongan
[INFO] Mengekstrak skill dari job descriptions ...


Resume job seeker

In [41]:
RESUMES = [
    {"name": "Andi Pratama", "resume": """
        Saya adalah lulusan S1 Informatika dengan pengalaman 2 tahun di bidang
        data analysis dan machine learning. Saya mahir menggunakan Python, pandas,
        numpy, dan scikit-learn untuk membangun model prediktif. Berpengalaman
        dalam visualisasi data menggunakan Tableau dan mengolah data dengan SQL.
        Memiliki pemahaman statistik yang baik dan terbiasa bekerja dengan
        dataset besar untuk menghasilkan insight bisnis.
    """},
    {"name": "Bunga Rahayu", "resume": """
        Lulusan S2 Manajemen dengan 4 tahun pengalaman sebagai project manager.
        Terbiasa memimpin tim lintas fungsi, menyusun anggaran, dan membuat
        laporan manajemen menggunakan Microsoft Office dan Excel. Memiliki
        kemampuan komunikasi dan leadership yang kuat. Berpengalaman dalam
        business analysis, budgeting, dan reporting kepada stakeholder senior.
    """},
    {"name": "Chandra Wijaya", "resume": """
        Fresh graduate S1 Teknik Elektro dengan pengalaman magang 6 bulan
        di perusahaan manufaktur. Memiliki keahlian dalam electrical design,
        AutoCAD, dan pemrograman PLC. Pernah menangani troubleshooting dan
        maintenance peralatan listrik di lini produksi. Familiar dengan
        mikrokontroler Arduino dan pengembangan sistem otomasi sederhana.
    """},
    {"name": "Dewi Kusuma", "resume": """
        Sarjana Akuntansi dengan 3 tahun pengalaman kerja di bidang keuangan
        dan perpajakan. Terampil dalam penyusunan laporan keuangan, audit
        internal, dan pengelolaan pajak menggunakan SAP dan Excel. Memiliki
        kemampuan analisis keuangan yang solid dan berpengalaman dalam budgeting.
    """},
    {"name": "Eko Santoso", "resume": """
        Software engineer dengan 5 tahun pengalaman membangun aplikasi web
        menggunakan JavaScript, React, dan Node.js. Terbiasa dengan pengembangan
        REST API, integrasi database SQL, dan deployment menggunakan Docker.
        Familiar dengan metodologi Agile dan version control menggunakan Git.
    """},
]

for r in RESUMES:
    r["clean"] = clean_text(r["resume"])
    r["skills"] = extract_skills(r["clean"])

print(f"[INFO] {len(RESUMES)} resume job seeker dimuat.")

print("[INFO] Membangun TF-IDF matrix ...")
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=15_000,
    stop_words=ALL_SW
)
tfidf_matrix = vectorizer.fit_transform(df_combined["clean_text"])
print(f"[INFO] TF-IDF: {tfidf_matrix.shape[0]:,} x {tfidf_matrix.shape[1]:,}")


[INFO] 5 resume job seeker dimuat.
[INFO] Membangun TF-IDF matrix ...
[INFO] TF-IDF: 20,000 x 15,000


# FEATURE ENGINEERING

In [42]:
section("FEATURE ENGINEERING")

print("""
Fitur yang dihitung per pasangan (resume, lowongan):
  1. cosine_sim         - TF-IDF cosine similarity
  2. skill_overlap_cnt  - jumlah skill resume yang ada di job desc
  3. skill_overlap_ratio - skill_overlap_cnt / total skill resume -> ground truth
  4. resume_skill_cnt   - total skill terdeteksi di resume
  5. job_skill_cnt      - total skill terdeteksi di job desc
  6. text_len_ratio     - panjang teks resume / panjang teks job desc
  7. category_match     - kecocokan kategori resume dan pekerjaan
""")

records = []

for seeker in RESUMES:
    name = seeker["name"]
    resume_skills = seeker["skills"]
    n_resume_skills = len(resume_skills) if resume_skills else 1

    resume_category = infer_resume_category(resume_skills)

    query_text = " ".join(resume_skills) if resume_skills else seeker["clean"]
    resume_vec = vectorizer.transform([query_text])
    sim_scores = cosine_similarity(resume_vec, tfidf_matrix).flatten()

    resume_len = len(seeker["clean"].split())

    for idx, row in df_combined.iterrows():
        job_skills = row["job_skills"]
        overlap_skills = set(resume_skills) & set(job_skills)
        overlap_cnt = len(overlap_skills)
        overlap_ratio = overlap_cnt / n_resume_skills

        job_len = len(str(row["clean_text"]).split())
        text_len_ratio = resume_len / max(job_len, 1)

        job_category = infer_job_category(row["job_title"])
        category_match = get_category_match(resume_category, job_category)

        records.append({
            "seeker": name,
            "job_idx": idx,
            "job_title": row["job_title"],
            "cosine_sim": float(sim_scores[idx]),
            "skill_overlap_cnt": overlap_cnt,
            "skill_overlap_ratio": overlap_ratio,
            "resume_skill_cnt": n_resume_skills,
            "job_skill_cnt": row["job_skill_count"],
            "text_len_ratio": text_len_ratio,
            "category_match": category_match,
        })

df_pairs = pd.DataFrame(records)
print(f"[INFO] Total pasangan (resume x lowongan): {len(df_pairs):,}")
print("\nDistribusi skill_overlap_ratio (target regresi):")
print(df_pairs["skill_overlap_ratio"].describe().round(4).to_string())

df_pairs["cosine_sim"] = df_pairs["cosine_sim"].clip(0, 1)
df_pairs["text_len_ratio"] = df_pairs["text_len_ratio"].clip(0, 5)
df_pairs["job_skill_cnt_log"] = np.log1p(df_pairs["job_skill_cnt"])
df_pairs["resume_skill_cnt_log"] = np.log1p(df_pairs["resume_skill_cnt"])



  FEATURE ENGINEERING

Fitur yang dihitung per pasangan (resume, lowongan):
  1. cosine_sim         - TF-IDF cosine similarity
  2. skill_overlap_cnt  - jumlah skill resume yang ada di job desc
  3. skill_overlap_ratio - skill_overlap_cnt / total skill resume -> ground truth
  4. resume_skill_cnt   - total skill terdeteksi di resume
  5. job_skill_cnt      - total skill terdeteksi di job desc
  6. text_len_ratio     - panjang teks resume / panjang teks job desc
  7. category_match     - kecocokan kategori resume dan pekerjaan

[INFO] Total pasangan (resume x lowongan): 100,000

Distribusi skill_overlap_ratio (target regresi):
count    100000.0000
mean          0.0559
std           0.1162
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0000
max           1.0000


# PEMBUATAN LABEL

In [43]:
section("PEMBUATAN LABEL")

RATIO_THRESH = 0.25
SIM_THRESH = 0.15

df_pairs["label"] = (
    (df_pairs["skill_overlap_ratio"] >= RATIO_THRESH) |
    (df_pairs["cosine_sim"] >= SIM_THRESH)
).astype(int)

n_pos = df_pairs["label"].sum()
n_neg = len(df_pairs) - n_pos

print("\nDistribusi label klasifikasi:")
print(f"  Cocok   (1) : {n_pos:>8,}  ({n_pos / len(df_pairs) * 100:.1f}%)")
print(f"  Tdk Cocok(0): {n_neg:>8,}  ({n_neg / len(df_pairs) * 100:.1f}%)")

print("\nTarget regresi (skill_overlap_ratio):")
print(f"  Mean  : {df_pairs['skill_overlap_ratio'].mean():.4f}")
print(f"  Std   : {df_pairs['skill_overlap_ratio'].std():.4f}")
print(f"  Max   : {df_pairs['skill_overlap_ratio'].max():.4f}")
print(f"  % = 0 : {(df_pairs['skill_overlap_ratio'] == 0).mean() * 100:.1f}%")

df_pairs["skill_ratio"] = df_pairs["skill_overlap_ratio"]

df_pairs["final_score"] = df_pairs.apply(
    lambda x: compute_final_score(
        x["cosine_sim"],
        x["skill_ratio"],
        x["category_match"]
    ),
    axis=1
)

top_jobs = df_pairs.sort_values(by="final_score", ascending=False).head(3)

print("\n[INFO] Menggunakan weighted scoring (similarity + skill + category)")
print("\n🔥 TOP 3 REKOMENDASI PEKERJAAN:")
print(top_jobs[["job_title", "final_score", "skill_ratio", "cosine_sim"]])


  PEMBUATAN LABEL

Distribusi label klasifikasi:
  Cocok   (1) :   10,495  (10.5%)
  Tdk Cocok(0):   89,505  (89.5%)

Target regresi (skill_overlap_ratio):
  Mean  : 0.0559
  Std   : 0.1162
  Max   : 1.0000
  % = 0 : 76.9%

[INFO] Menggunakan weighted scoring (similarity + skill + category)

🔥 TOP 3 REKOMENDASI PEKERJAAN:
                             job_title  final_score  skill_ratio  cosine_sim
2500      Machine Learning Scientist I     0.565569     0.857143    0.335472
86891  Senior Software Engineer (.Net)     0.506075     1.000000    0.176792
11040                   Data Scientist     0.491739     0.714286    0.271947


# TRAIN/TEST SPLIT

In [44]:
section("TRAIN/TEST SPLIT")

FEATURES = [
    "cosine_sim",
    "skill_overlap_cnt",
    "resume_skill_cnt",
    "job_skill_cnt",
    "text_len_ratio",
    "category_match"
]

X = df_pairs[FEATURES].values
y_cls = df_pairs["label"].values
y_reg = df_pairs["skill_overlap_ratio"].values

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_cls, y_reg, test_size=0.2, random_state=42, stratify=y_cls
)

print(f"  Train : {len(X_train):,} pasangan")
print(f"  Test  : {len(X_test):,} pasangan")
print(f"  Fitur : {X.shape[1]} ({', '.join(FEATURES)})")



  TRAIN/TEST SPLIT
  Train : 80,000 pasangan
  Test  : 20,000 pasangan
  Fitur : 6 (cosine_sim, skill_overlap_cnt, resume_skill_cnt, job_skill_cnt, text_len_ratio, category_match)


# CLASSIFICATION MODEL

In [45]:
section("CLASSIFICATION MODEL  (Accuracy >= 85%)")


  CLASSIFICATION MODEL  (Accuracy >= 85%)


A.Logistic Regression

In [46]:
sub("A.Logistic Regression")

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=500,
        class_weight="balanced",
        random_state=42
    ))
])

pipe_lr.fit(X_train, yc_train)
yc_pred_lr = pipe_lr.predict(X_test)

acc_lr = accuracy_score(yc_test, yc_pred_lr)
prec_lr = precision_score(yc_test, yc_pred_lr, zero_division=0)
rec_lr = recall_score(yc_test, yc_pred_lr, zero_division=0)
f1_lr = f1_score(yc_test, yc_pred_lr, zero_division=0)

print(f"  Accuracy  : {acc_lr:.4f}  ({'[OK] >= 85%' if acc_lr >= 0.85 else '[X] < 85%'})")
print(f"  Precision : {prec_lr:.4f}")
print(f"  Recall    : {rec_lr:.4f}")
print(f"  F1-Score  : {f1_lr:.4f}")


------------------------------------------------------------
  A.Logistic Regression
------------------------------------------------------------
  Accuracy  : 0.9982  ([OK] >= 85%)
  Precision : 0.9831
  Recall    : 1.0000
  F1-Score  : 0.9915


B.Random Forest Classifier

In [47]:
sub("B.Random Forest Classifier")
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])
pipe_rf.fit(X_train, yc_train)
yc_pred_rf = pipe_rf.predict(X_test)

acc_rf = accuracy_score(yc_test, yc_pred_rf)
prec_rf = precision_score(yc_test, yc_pred_rf, zero_division=0)
rec_rf = recall_score(yc_test, yc_pred_rf, zero_division=0)
f1_rf = f1_score(yc_test, yc_pred_rf, zero_division=0)

print(f"  Accuracy  : {acc_rf:.4f}  ({'[OK] >= 85%' if acc_rf >= 0.85 else '[X] < 85%'})")
print(f"  Precision : {prec_rf:.4f}")
print(f"  Recall    : {rec_rf:.4f}")
print(f"  F1-Score  : {f1_rf:.4f}")

best_clf = pipe_rf if acc_rf >= acc_lr else pipe_lr
best_clf_name = "Random Forest" if acc_rf >= acc_lr else "Logistic Regression"
best_acc = max(acc_rf, acc_lr)

print(f"\n  Model terbaik: {best_clf_name}  (Accuracy = {best_acc:.4f})")

print(f"\n  Classification Report ({best_clf_name}):")
yc_pred_best = best_clf.predict(X_test)
print(classification_report(
    yc_test, yc_pred_best,
    target_names=["Tidak Cocok", "Cocok"],
    zero_division=0
))


------------------------------------------------------------
  B.Random Forest Classifier
------------------------------------------------------------
  Accuracy  : 1.0000  ([OK] >= 85%)
  Precision : 0.9995
  Recall    : 1.0000
  F1-Score  : 0.9998

  Model terbaik: Random Forest  (Accuracy = 1.0000)

  Classification Report (Random Forest):
              precision    recall  f1-score   support

 Tidak Cocok       1.00      1.00      1.00     17901
       Cocok       1.00      1.00      1.00      2099

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



# REGRESSION MODEL

In [48]:
section("REGRESSION MODEL  (MAE <= 0.02)")


  REGRESSION MODEL  (MAE <= 0.02)


A.Ridge Regression

In [49]:
sub("A.Ridge Regression")
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=0.5)),
])
pipe_ridge.fit(X_train, yr_train)
yr_pred_ridge = pipe_ridge.predict(X_test).clip(0, 1)

mae_ridge = mean_absolute_error(yr_test, yr_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(yr_test, yr_pred_ridge))
r2_ridge = r2_score(yr_test, yr_pred_ridge)

print(f"  MAE   : {mae_ridge:.4f}  ({'[OK] <= 0.02' if mae_ridge <= 0.02 else '[X] > 0.02'})")
print(f"  RMSE  : {rmse_ridge:.4f}")
print(f"  R2    : {r2_ridge:.4f}")


------------------------------------------------------------
  A.Ridge Regression
------------------------------------------------------------
  MAE   : 0.0097  ([OK] <= 0.02)
  RMSE  : 0.0211
  R2    : 0.9671


B.Gradient Boosting Regressor

In [50]:
sub("B.Gradient Boosting Regressor")
pipe_gbr = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", GradientBoostingRegressor(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
    )),
])
pipe_gbr.fit(X_train, yr_train)
yr_pred_gbr = pipe_gbr.predict(X_test).clip(0, 1)

mae_gbr = mean_absolute_error(yr_test, yr_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(yr_test, yr_pred_gbr))
r2_gbr = r2_score(yr_test, yr_pred_gbr)

print(f"  MAE   : {mae_gbr:.4f}  ({'[OK] <= 0.02' if mae_gbr <= 0.02 else '[X] > 0.02'})")
print(f"  RMSE  : {rmse_gbr:.4f}")
print(f"  R2    : {r2_gbr:.4f}")

best_reg = pipe_gbr if mae_gbr <= mae_ridge else pipe_ridge
best_reg_name = "Gradient Boosting" if mae_gbr <= mae_ridge else "Ridge Regression"
best_mae = min(mae_gbr, mae_ridge)
yr_pred_best = pipe_gbr.predict(X_test).clip(0, 1) if mae_gbr <= mae_ridge else yr_pred_ridge

print(f"\n  Model terbaik: {best_reg_name}  (MAE = {best_mae:.4f})")


------------------------------------------------------------
  B.Gradient Boosting Regressor
------------------------------------------------------------
  MAE   : 0.0000  ([OK] <= 0.02)
  RMSE  : 0.0004
  R2    : 1.0000

  Model terbaik: Gradient Boosting  (MAE = 0.0000)


# VISUALISASI HASIL MODEL

In [51]:
section("VISUALISASI HASIL MODEL")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_cls = ["Logistic\nRegression", "Random\nForest"]
accs = [acc_lr, acc_rf]
bar_colors = [ACCENT2 if a >= 0.85 else PALETTE[2] for a in accs]
bars = axes[0].bar(models_cls, accs, color=bar_colors, edgecolor="none", width=0.4)
axes[0].axhline(0.85, color=PALETTE[2], linestyle="--", linewidth=2, label="Target: 85%")
for b, a in zip(bars, accs):
    axes[0].text(
        b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
        f"{a:.3f}", ha="center", va="bottom", color=TEXT_MAIN,
        fontsize=12, fontweight="bold"
    )
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Accuracy", color=TEXT_MAIN)
axes[0].set_title("Accuracy - Classification Models", color=TEXT_MAIN, fontsize=12, fontweight="bold")
axes[0].legend(facecolor=BG_MID, edgecolor="#30363D", labelcolor=TEXT_MAIN)
axes[0].set_facecolor(BG_MID)

cm = confusion_matrix(yc_test, yc_pred_best)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Tidak Cocok", "Cocok"],
    yticklabels=["Tidak Cocok", "Cocok"],
    linewidths=0.5, linecolor=BG_DARK, ax=axes[1],
    cbar_kws={"shrink": 0.8}
)
axes[1].set_title(f"Confusion Matrix - {best_clf_name}", color=TEXT_MAIN, fontsize=12, fontweight="bold")
axes[1].tick_params(colors=TEXT_MAIN, labelsize=9)
axes[1].set_facecolor(BG_MID)

rf_model = pipe_rf.named_steps.get("clf", None)

if rf_model is not None:
    importances = rf_model.feature_importances_
    feat_df = pd.DataFrame({"Feature": FEATURES, "Importance": importances}).sort_values("Importance", ascending=True)
    axes[2].barh(feat_df["Feature"], feat_df["Importance"], color=PALETTE[:len(FEATURES)], edgecolor="none")
    axes[2].set_title("Feature Importance - Random Forest", color=TEXT_MAIN, fontsize=12, fontweight="bold")
    axes[2].set_xlabel("Importance", color=TEXT_MAIN)
    axes[2].set_facecolor(BG_MID)
    axes[2].tick_params(labelsize=9)

fig.patch.set_facecolor(BG_DARK)
plt.tight_layout()
plt.savefig("plots/model_01_classification.png", dpi=150, bbox_inches="tight")
print("[OK] Disimpan: plots/model_01_classification.png")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_reg = ["Ridge\nRegression", "Gradient\nBoosting"]
maes = [mae_ridge, mae_gbr]
bar_clrs2 = [ACCENT2 if m <= 0.02 else PALETTE[2] for m in maes]
bars2 = axes[0].bar(models_reg, maes, color=bar_clrs2, edgecolor="none", width=0.4)
axes[0].axhline(0.02, color=PALETTE[2], linestyle="--", linewidth=2, label="Target: MAE=0.02")
for b, m in zip(bars2, maes):
    axes[0].text(
        b.get_x() + b.get_width() / 2, b.get_height() + 0.0003,
        f"{m:.4f}", ha="center", va="bottom", color=TEXT_MAIN,
        fontsize=12, fontweight="bold"
    )
axes[0].set_ylabel("MAE", color=TEXT_MAIN)
axes[0].set_title("MAE - Regression Models", color=TEXT_MAIN, fontsize=12, fontweight="bold")
axes[0].legend(facecolor=BG_MID, edgecolor="#30363D", labelcolor=TEXT_MAIN)
axes[0].set_facecolor(BG_MID)

sample_idx = np.random.choice(len(yr_test), min(2000, len(yr_test)), replace=False)
axes[1].scatter(yr_test[sample_idx], yr_pred_best[sample_idx], color=ACCENT, alpha=0.3, s=8, edgecolors="none")
lim = max(yr_test.max(), yr_pred_best.max()) * 1.05
axes[1].plot([0, lim], [0, lim], color=PALETTE[2], linestyle="--", linewidth=1.5, label="Perfect prediction")
axes[1].set_xlabel("Actual (skill_overlap_ratio)", color=TEXT_MAIN)
axes[1].set_ylabel("Predicted", color=TEXT_MAIN)
axes[1].set_title(f"Actual vs Predicted - {best_reg_name}", color=TEXT_MAIN, fontsize=12, fontweight="bold")
axes[1].legend(facecolor=BG_MID, edgecolor="#30363D", labelcolor=TEXT_MAIN, fontsize=9)
axes[1].set_facecolor(BG_MID)

residuals = yr_test - yr_pred_best
axes[2].hist(residuals, bins=60, color=ACCENT, edgecolor="none", alpha=0.85)
axes[2].axvline(0, color=PALETTE[2], linestyle="--", linewidth=2)
axes[2].axvline(residuals.mean(), color=PALETTE[3], linestyle="-.", linewidth=1.5, label=f"Mean={residuals.mean():.4f}")
axes[2].set_xlabel("Residual (Actual - Predicted)", color=TEXT_MAIN)
axes[2].set_ylabel("Frekuensi", color=TEXT_MAIN)
axes[2].set_title("Distribusi Residual", color=TEXT_MAIN, fontsize=12, fontweight="bold")
axes[2].legend(facecolor=BG_MID, edgecolor="#30363D", labelcolor=TEXT_MAIN, fontsize=9)
axes[2].set_facecolor(BG_MID)

fig.patch.set_facecolor(BG_DARK)
plt.tight_layout()
plt.savefig("plots/model_02_regression.png", dpi=150, bbox_inches="tight")
print("[OK] Disimpan: plots/model_02_regression.png")
plt.close()



  VISUALISASI HASIL MODEL
[OK] Disimpan: plots/model_01_classification.png
[OK] Disimpan: plots/model_02_regression.png


# TOP 3 JOB PER RESUME

In [52]:
section("TOP 3 JOB PER RESUME")

print("""
Alur prediksi:
  1. Ekstrak skill dari resume mentah
  2. Hitung cosine_sim antara resume dan setiap lowongan
  3. Hitung skill_overlap_ratio, dll.
  4. Classification: prediksi apakah pasangan cocok (1) atau tidak (0)
  5. Filter hanya yang diprediksi cocok
  6. Regression: prediksi skor kesesuaian (0-1)
  7. Urutkan descending -> ambil Top 3
""")

all_top3 = {}

for seeker in RESUMES:
    name = seeker["name"]
    resume_clean = seeker["clean"]
    resume_skills = seeker["skills"]
    n_rs = len(resume_skills) if resume_skills else 1

    query_text = " ".join(resume_skills) if resume_skills else resume_clean
    resume_vec = vectorizer.transform([query_text])
    sim_scores = cosine_similarity(resume_vec, tfidf_matrix).flatten()

    resume_category = infer_resume_category(resume_skills)
    resume_len = len(resume_clean.split())

    feats = []

    for idx, row in df_combined.iterrows():
        js = row["job_skills"]

        overlap = len(set(resume_skills) & set(js))
        skill_ratio = overlap / n_rs

        jlen = len(str(row["clean_text"]).split())

        job_category = infer_job_category(row["job_title"])
        category_match = get_category_match(resume_category, job_category)

        feats.append([
            float(sim_scores[idx]),
            overlap,
            n_rs,
            row["job_skill_count"],
            resume_len / max(jlen, 1),
            category_match,
        ])

    X_pred = np.array(feats)
    y_class_pred = best_clf.predict(X_pred)

    match_idx = np.where(y_class_pred == 1)[0]

    if len(match_idx) == 0:
        print(f"\n[{name}] Tidak ada lowongan yang diprediksi cocok oleh classifier.")
        continue

    scores_final = []

    for idx in match_idx:
        overlap = len(set(resume_skills) & set(df_combined.iloc[idx]["job_skills"]))
        skill_ratio = overlap / n_rs

        score = best_reg.predict([[
            sim_scores[idx],
            overlap,
            n_rs,
            df_combined.iloc[idx]["job_skill_count"],
            resume_len / max(len(str(df_combined.iloc[idx]["clean_text"]).split()), 1),
            feats[idx][-1]
        ]])[0]

        scores_final.append(score)

    scores_final = np.array(scores_final)
    top3_local_idx = np.argsort(scores_final)[::-1][:3]
    top3_global_idx = match_idx[top3_local_idx]
    top3_df = df_combined.iloc[top3_global_idx].copy()
    top3_df["predicted_score"] = scores_final[top3_local_idx]
    top3_df["cosine_sim"] = sim_scores[top3_global_idx]
    top3_df = top3_df.reset_index(drop=True)
    top3_df.index += 1

    all_top3[name] = top3_df

    resume_text = " ".join(seeker["resume"].split())
    resume_wrapped = textwrap.fill(resume_text, width=56)
    resume_ind = "\n".join("    " + l for l in resume_wrapped.splitlines())

    print(f"\n{'=' * 60}")
    print(f"  Job Seeker       : {name}")
    print("  Resume Mentah    :")
    print(resume_ind)
    print(f"  Skill Terdeteksi : {', '.join(resume_skills) if resume_skills else '-'}")
    print(f"  Kandidat cocok (classifier=1): {len(match_idx):,} lowongan")
    print(f"{'=' * 60}")

    for rank, row in top3_df.iterrows():
            exp = row["experience_level"] if pd.notna(row["experience_level"]) else "N/A"
            wt = row["work_type"] if pd.notna(row["work_type"]) else "N/A"
            loc = row["location"] if pd.notna(row["location"]) else "N/A"
            cmp = row.get("company", "N/A")
            cmp = cmp if pd.notna(cmp) else "N/A"

            print(f"  [Rank {rank}] [OK] {row['job_title']}")
            print(f"           Predicted Score : {row['predicted_score']:.4f}")
            print(f"           Cosine Sim      : {row['cosine_sim']:.4f}")
            print(f"           Perusahaan      : {cmp}")
            print(f"           Level           : {exp} | Tipe: {wt}")
            print(f"           Lokasi          : {loc} | Sumber: {row['source']}")
            print()

n_seekers = len(all_top3)
fig, axes = plt.subplots(1, n_seekers, figsize=(5 * n_seekers, 6), sharey=False)
if n_seekers == 1:
    axes = [axes]

for ax, (name, top3_df) in zip(axes, all_top3.items()):
    labels = [textwrap.fill(t, 18) for t in top3_df["job_title"].values]
    scores = top3_df["predicted_score"].values
    bars_t = ax.bar(["Rank 1", "Rank 2", "Rank 3"], scores, color=PALETTE[:3], edgecolor="none", width=0.5)

    for bw, lbl in zip(bars_t, labels):
        ax.text(
            bw.get_x() + bw.get_width() / 2,
            bw.get_height() + scores.max() * 0.01,
            f"{bw.get_height():.4f}",
            ha="center", va="bottom", color=TEXT_MAIN, fontsize=8
        )
        ax.text(
            bw.get_x() + bw.get_width() / 2,
            bw.get_height() / 2,
            lbl,
            ha="center", va="center", color=BG_DARK, fontsize=7, fontweight="bold"
        )

    ax.set_title(name.split()[0], color=TEXT_MAIN, fontsize=10, fontweight="bold")
    ax.set_ylim(0, scores.max() * 1.3)
    ax.set_ylabel("Predicted Score (Regression)", color=TEXT_MAIN, fontsize=8)
    ax.set_facecolor(BG_MID)
    ax.tick_params(labelsize=9)

fig.patch.set_facecolor(BG_DARK)
fig.suptitle("Top 3 Job Recommendations - ML Model", color=TEXT_MAIN, fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout(pad=2)
plt.savefig("plots/model_03_top3_predictions.png", dpi=150, bbox_inches="tight")
print("[OK] Disimpan: plots/model_03_top3_predictions.png")
plt.close()



  TOP 3 JOB PER RESUME

Alur prediksi:
  1. Ekstrak skill dari resume mentah
  2. Hitung cosine_sim antara resume dan setiap lowongan
  3. Hitung skill_overlap_ratio, dll.
  4. Classification: prediksi apakah pasangan cocok (1) atau tidak (0)
  5. Filter hanya yang diprediksi cocok
  6. Regression: prediksi skor kesesuaian (0-1)
  7. Urutkan descending -> ambil Top 3


  Job Seeker       : Andi Pratama
  Resume Mentah    :
    Saya adalah lulusan S1 Informatika dengan pengalaman 2
    tahun di bidang data analysis dan machine learning. Saya
    mahir menggunakan Python, pandas, numpy, dan scikit-
    learn untuk membangun model prediktif. Berpengalaman
    dalam visualisasi data menggunakan Tableau dan mengolah
    data dengan SQL. Memiliki pemahaman statistik yang baik
    dan terbiasa bekerja dengan dataset besar untuk
    menghasilkan insight bisnis.
  Skill Terdeteksi : python, sql, machine learning, pandas, numpy, data analysis, tableau
  Kandidat cocok (classifier=1): 431 lowong

# EVALUASI MODEL

In [53]:
section("EVALUASI MODEL")

passed_acc = best_acc >= 0.85
passed_mae = best_mae <= 0.02

print("\n" + "=" * 70)
print("HASIL EVALUASI MODEL JOB MATCHING")
print("=" * 70)
print(f"Accuracy target >= 85.00% : {best_acc * 100:6.2f}%   {'[OK] TERCAPAI' if passed_acc else '[X] BELUM'}")
print(f"MAE target <= 0.0200      : {best_mae:8.4f}   {'[OK] TERCAPAI' if passed_mae else '[X] BELUM'}")
print(f"Model Klasifikasi         : {best_clf_name}")
print(f"Model Regresi             : {best_reg_name}")
print("Ground Truth              : skill_overlap_ratio")
print("Binary Label              : overlap_ratio >= 0.25 OR cosine_sim >= 0.15")
print("Output                    : Top 3 Job Title per resume")

print("\n[FILE VISUALISASI]")
print("  plots/model_01_classification.png - Accuracy, Confusion Matrix, Feature Importance")
print("  plots/model_02_regression.png - MAE, Actual vs Predicted, Residual")
print("  plots/model_03_top3_predictions.png - Top 3 Rekomendasi per Resume")

joblib.dump(best_clf, "saved_model/classifier.pkl")
joblib.dump(best_reg, "saved_model/regressor.pkl")
joblib.dump(vectorizer, "saved_model/tfidf_vectorizer.pkl")
joblib.dump(SKILLS, "saved_model/skills.pkl")

print("  - saved_model/classifier.pkl")
print("  - saved_model/regressor.pkl")
print("  - saved_model/tfidf_vectorizer.pkl")
print("  - saved_model/skills.pkl")
print(f"\n{'=' * 70}")
print("  MODEL SELESAI")
print(f"{'=' * 70}\n")


  EVALUASI MODEL

HASIL EVALUASI MODEL JOB MATCHING
Accuracy target >= 85.00% : 100.00%   [OK] TERCAPAI
MAE target <= 0.0200      :   0.0000   [OK] TERCAPAI
Model Klasifikasi         : Random Forest
Model Regresi             : Gradient Boosting
Ground Truth              : skill_overlap_ratio
Binary Label              : overlap_ratio >= 0.25 OR cosine_sim >= 0.15
Output                    : Top 3 Job Title per resume

[FILE VISUALISASI]
  plots/model_01_classification.png - Accuracy, Confusion Matrix, Feature Importance
  plots/model_02_regression.png - MAE, Actual vs Predicted, Residual
  plots/model_03_top3_predictions.png - Top 3 Rekomendasi per Resume
  - saved_model/classifier.pkl
  - saved_model/regressor.pkl
  - saved_model/tfidf_vectorizer.pkl
  - saved_model/skills.pkl

  MODEL SELESAI

